# Educational conversion propensity model

This notebook evaluates an identifier-free, one-row-per-session sample from Google's public GA4 ecommerce dataset. It is educational decision support only: it is not a causal analysis, a production targeting system, or a claim about campaign performance.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))
from marketing_measurement.modeling.conversion import (
    evaluate_conversion_model,
    train_conversion_model,
)

data_path = project_root / 'data/observed/ga4_public_sample/conversion_model_sessions.json.gz'
sessions = pd.read_json(data_path)
sessions.shape, sessions['converted'].value_counts(dropna=False).to_dict()

## Leakage boundary

The model has an exact pre-outcome allowlist: session-start calendar fields, device, coarse geography, new/returning status, and acquisition fields captured on the session's first observed event. Every other requested field is rejected, including unknown aliases, targets, IDs, split/provenance fields, engagement, cart, checkout, purchase/revenue, transaction data, duration, and later page behavior. `user_group_bucket` is not a feature: it is a non-unique deterministic split key.

In [ ]:
features = sessions.drop(columns='converted')
bundle = train_conversion_model(
    features=features,
    target=sessions['converted'],
    groups=sessions['user_group_bucket'],
)
report = evaluate_conversion_model(bundle, bundle.test)

{
    'selected_model': bundle.selected_model_name,
    'rows': len(sessions),
    'overall_prevalence': float(sessions['converted'].eq('true').mean()),
    'training_prevalence': bundle.training_prevalence,
    'split': bundle.split_metadata,
    'cross_validation': bundle.cv_scores,
}

In [ ]:
pd.DataFrame(report.model_metrics).T.assign(
    baseline_pr_auc=report.baseline_pr_auc,
    baseline_roc_auc=report.baseline_roc_auc,
    baseline_brier_score=report.baseline_brier_score,
)

In [ ]:
model_evaluation = {
    'selected_model': bundle.selected_model_name,
    'evaluation_population': f"{len(bundle.test.target):,} held-out identifier-free public-sample sessions",
    'metrics': {
        'no_skill_held_out': {
            'roc_auc': round(report.baseline_roc_auc, 6),
            'pr_auc': round(report.baseline_pr_auc, 6),
            'brier_score': round(report.baseline_brier_score, 6),
        },
        'logistic_regression_training_cv': {
            metric: round(value, 6)
            for metric, value in bundle.cv_scores['logistic_regression'].items()
        },
        'logistic_regression_held_out': {
            metric: round(value, 6)
            for metric, value in report.model_metrics['logistic_regression'].items()
        },
        'random_forest_held_out': {
            metric: round(value, 6)
            for metric, value in report.model_metrics['random_forest'].items()
        },
    },
    'selected_threshold': round(report.selected_threshold, 6),
    'threshold_label': report.selected_threshold_label,
}
output_path = project_root / 'data/derived/ga4_public_sample/conversion_model_evaluation.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(model_evaluation, indent=2, sort_keys=True) + '\n')
report.threshold_table, report.confusion_matrix, report.calibration_bins, report.subgroup_diagnostics

## Calibration decision

No probability calibration is applied. The untouched holdout is used only for diagnosis; it must not be reused to fit a calibrator. The Brier score and five-bin calibration evidence are reported for the selected uncalibrated model. A future model could use a separately nested, group-disjoint training validation process if calibration were justified.